# AgroguardAI-LLM Training – Llama‑3.2‑3B + Nigeria Dataset

Fine‑tune `meta-llama/Llama-3.2-3B-Instruct` on the **Nigeria‑only** AgroguardAI‑LLM dataset (Hausa, Igbo, Yoruba, Fulfulde) using QLoRA.

Runs on a free **T4 GPU** (16 GB VRAM). The training script (`scripts/finetune_llama3.py`) reads your `configs/llama3_qlora_v1.json` and applies the `llama3‑3b` preset.

**Before running:** Mount Google Drive and paste your Hugging Face token to push the trained adapter to the Hub.

In [ ]:
# @title 1. Install dependencies
!pip install -qU transformers accelerate bitsandbytes peft datasets huggingface_hub
!pip install -qU xformers --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# @title 2. Mount Google Drive (saves adapter here)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/agroguardai'
!mkdir -p {DRIVE_PATH}

In [ ]:
# @title 3. Clone the repo and switch to the main branch
!git clone https://github.com/agroguardaiaOS/agroguardai-llm.git /content/agroguardai-llm
%cd /content/agroguardai-llm
!git checkout main

In [ ]:
# @title 4. Prepare dataset (Nigeria‑only)
# Assumes the training and test files are already split and saved:
# data/processed/train_nigeria.json   data/processed/test_nigeria.json
# If you need to recreate splits, uncomment the next block

!python -c "import json; d=json.load(open('data/processed/train_nigeria.json')); print(f'Training entries: {len(d)}')"
!python -c "import json; d=json.load(open('data/processed/test_nigeria.json')); print(f'Test entries: {len(d)}')"

In [ ]:
# @title 5. Run QLoRA fine‑tuning (Llama‑3‑3B, T4 preset)
HF_TOKEN = ""  # @param {type:"string"}
# Paste your Hugging Face token if you want to push the adapter to the Hub.

import os
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face Hub")
else:
    print("No HF_TOKEN provided – adapter saved locally only.")

# The training script uses the llama3‑3b preset from configs/llama3_qlora_v1.json
# If your Nigeria‑only dataset is in train_nigeria.json/test_nigeria.json,
# we pass them explicitly via command line.

!python scripts/finetune_llama3.py \
    --model-family llama3-3b \
    --train-file data/processed/train_nigeria.json \
    --val-file data/processed/test_nigeria.json \
    --output ./models/llama3-agricultural-qlora

In [ ]:
# @title 6. Save adapter to Google Drive
import shutil
adapter_dir = '/content/agroguardai-llm/models/llama3-agricultural-qlora'
if os.path.exists(adapter_dir):
    dest = os.path.join(DRIVE_PATH, 'llama3-agricultural-qlora')
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(adapter_dir, dest)
    print(f'Adapter copied to {dest}')
else:
    print('Training did not produce an adapter. Check logs.')

In [ ]:
# @title 7. (Optional) Push adapter to Hugging Face Hub
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi()
    repo_id = "AgroguardAI/llama3-agri-qlora"
    api.upload_folder(
        folder_path=adapter_dir,
        repo_id=repo_id,
        repo_type="model",
        commit_message="QLoRA adapter for Llama‑3‑3B on Nigeria‑only agricultural dataset"
    )
    print(f'Adapter pushed to https://huggingface.co/{repo_id}')
else:
    print('No HF_TOKEN provided – adapter not pushed.')

In [ ]:
# @title 8. Evaluate with benchmark (AgriBench / Nigeria‑only)
!python scripts/evaluate_benchmark.py \
    --adapter ./models/llama3-agricultural-qlora \
    --base meta-llama/Llama-3.2-3B-Instruct \
    --test-file data/processed/test_nigeria.json \
    --output results/benchmark_nigeria.json

## After training
- The fine‑tuned adapter is saved in your Google Drive under `agroguardai/llama3-agricultural-qlora`.
- If you provided your HF token, it was pushed to `https://huggingface.co/AgroguardAI/llama3-agri-qlora`.
- Benchmark results are in `results/benchmark_nigeria.json`.
- Update your `configs/llama3_qlora_v1.json` with the new dataset stats and model name.